In [1]:
import pandas as pd
import numpy as np
import re
import os
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# Paths
DATA_DIR = "data/"
OUTPUT_DIR = "outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

YEARS = [2023, 2024, 2025]

In [3]:
# Fonctions
def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]+", "_", regex=True)
    )
    return df

def normalize_text(s):
    if pd.isna(s):
        return s
    return str(s).strip().lower()

def filter_years(df):
    if "year" in df.columns:
        df = df[df["year"].isin(YEARS)]
    return df

def safe_div(a, b):
    return np.where((b == 0) | (pd.isna(b)), np.nan, a / b)

In [4]:
# Import des fichiers
from pathlib import Path

finance = pd.read_excel("AlbertSchool_CACEIS_PL-FTE_22-25_Sent.xlsx")

training = pd.read_excel("Training_Records_Unnamed.xlsx")

cold_review = pd.read_excel("Cold_Review_Unnamed.xlsx")

absenteeism_2025 = pd.read_excel("20260121 - Absentéisme_-_détail_affectation_-_Bilan_social 2025.xlsx", sheet_name="extract")
absenteeism_2024 = pd.read_excel("Absentéisme_-_détail_affectation_-_Bilan_social_2024.xlsx", sheet_name="Rapport 1", header=1)
absenteeism_2023 = pd.read_excel("Absentéisme_-_détail_affectation_-_Bilan_social_2023.xlsx", sheet_name="Rapport 1", header=1)


In [5]:
# Nettoyage des colonnes
finance = clean_columns(finance)
training = clean_columns(training)
cold_review = clean_columns(cold_review)
absenteeism_2025 = clean_columns(absenteeism_2025)
absenteeism_2024 = clean_columns(absenteeism_2024)
absenteeism_2023 = clean_columns(absenteeism_2023) 

In [6]:
print("Finance columns:", finance.columns.tolist())
print("Training columns:", training.columns.tolist())


Finance columns: ['toutes_filiales_conso', 'réel_décembre_2022', 'réel_décembre_2023', 'réel_décembre_2024', 'réel_décembre_2025', 'unnamed_5', 'o_w_europe', 'réel_décembre_2022_1', 'réel_décembre_2023_1', 'réel_décembre_2024_1', 'réel_décembre_2025_1']
Training columns: ['employee_code', 'entity', 'direction', 'attended_courses', 'organization', 'seesion_start_date', 'session_end_date', 'session_id', 'status', 'total_training_hours', 'certifications', 'year']


### HCVA computation

In [7]:
# fichier Finance : On sépare les deux blocs visibles :
# - Toutes Filiales Conso
# - O/W Europe

finance_raw = finance.copy()

finance_raw.head(10)

,toutes_filiales_conso,réel_décembre_2022,réel_décembre_2023,réel_décembre_2024,réel_décembre_2025,unnamed_5,o_w_europe,réel_décembre_2022_1,réel_décembre_2023_1,réel_décembre_2024_1,réel_décembre_2025_1
0,Net Commission Income,8.986520e+05,1.039248e+06,1.296794e+06,1.310191e+06,NaN,Net Commission Income,8.986520e+05,1.039441e+06,1.298118e+06,1.310091e+06
1,Net Interest Margin,4.186306e+05,6.944518e+05,8.265549e+05,8.676323e+05,NaN,Net Interest Margin,4.186306e+05,6.942633e+05,8.265514e+05,8.681490e+05
2,Other Income / (expense),-6.731757e+04,-5.636783e+04,-3.991230e+04,-7.781260e+04,NaN,Other Income / (expense),-6.731757e+04,-5.900319e+04,-4.145521e+04,-7.782268e+04
3,Net Banking Income (PNB),1.249965e+06,1.677332e+06,2.083437e+06,2.100011e+06,NaN,Net Banking Income (PNB),1.249965e+06,1.674701e+06,2.083214e+06,2.100417e+06
4,Rémunérations & charges,-3.923482e+05,-5.149449e+05,-6.580353e+05,-6.787855e+05,NaN,Rémunérations & charges,-3.923482e+05,-5.026284e+05,-6.309239e+05,-6.479100e+05
5,Recrutement,-3.141853e+03,-3.117929e+03,-3.199803e+03,-2.618171e+03,NaN,Recrutement,-3.141853e+03,-2.931983e+03,-2.860442e+03,-2.618171e+03
6,Formation (training costs),-4.385546e+03,-6.282893e+03,-5.924728e+03,-5.118793e+03,NaN,Formation (training costs),-4.385546e+03,-6.101923e+03,-5.654003e+03,-5.118793e+03
7,Other personnel costs,-6.056440e+04,-7.761735e+04,-9.776850e+04,-1.036153e+05,NaN,Other personnel costs,-6.056440e+04,-7.526670e+04,-9.172738e+04,-9.652365e+04
8,Total Personnel Costs,-4.604400e+05,-6.019631e+05,-7.649284e+05,-7.901378e+05,NaN,Total Personnel Costs,-4.604400e+05,-5.869290e+05,-7.311658e+05,-7.521706e+05
9,Other Operating Costs,-4.350758e+05,-5.814016e+05,-7.152383e+05,-6.502823e+05,NaN,Other Operating Costs,-4.350758e+05,-5.948595e+05,-7.520772e+05,-6.910448e+05


In [8]:
finance.columns = [
    "metric_conso",
    "2022_conso",
    "2023_conso",
    "2024_conso",
    "2025_conso",
    "separator",
    "metric_europe",
    "2022_europe",
    "2023_europe",
    "2024_europe",
    "2025_europe"
]

In [9]:
conso = finance[
    ["metric_conso", "2023_conso", "2024_conso", "2025_conso"]
].copy()

conso = conso.rename(columns={"metric_conso": "metric"})

conso["legal_entity"] = "Toutes Filiales Conso"

In [10]:
europe = finance[
    ["metric_europe", "2023_europe", "2024_europe", "2025_europe"]
].copy()

europe = europe.rename(columns={"metric_europe": "metric"})

europe["legal_entity"] = "O/W Europe"

In [11]:
def quick_melt(df, suffix):
    years = [f"2023_{suffix}", f"2024_{suffix}", f"2025_{suffix}"]
    return df.melt(
        id_vars=["legal_entity", "metric"],
        value_vars=years,
        var_name="year",
        value_name="value"
    )

finance_tidy = pd.concat([quick_melt(conso, "conso"), quick_melt(europe, "europe")], ignore_index=True)

# Extraction de l'année et nettoyage en une fois
finance_tidy["year"] = finance_tidy["year"].str.extract(r"(\d{4})").astype(int)

finance_tidy["value"] = pd.to_numeric(
    finance_tidy["value"]
    .astype(str)
    .str.replace(r"[\s\xa0]", "", regex=True) # Supprime tous les types d'espaces (y compris insécables)
    .str.replace(",", "."), 
    errors="coerce"
)

finance_tidy.sample(10)

,legal_entity,metric,year,value
30,Toutes Filiales Conso,Formation (training costs),2025,-5.118793e+03
64,O/W Europe,Rémunérations & charges,2025,-6.479100e+05
58,O/W Europe,Total Operating Costs,2024,-1.483243e+06
35,Toutes Filiales Conso,Gross Operating Income (RBE),2025,6.595905e+05
23,Toutes Filiales Conso,Gross Operating Income (RBE),2024,6.032700e+05
46,O/W Europe,Total Operating Costs,2023,-1.181789e+06
31,Toutes Filiales Conso,Other personnel costs,2025,-1.036153e+05
43,O/W Europe,Other personnel costs,2023,-7.526670e+04
26,Toutes Filiales Conso,Other Income / (expense),2025,-7.781260e+04
28,Toutes Filiales Conso,Rémunérations & charges,2025,-6.787855e+05


In [12]:
hcva_input = finance_tidy[
    finance_tidy["metric"].isin([
        "Net Banking Income (PNB)",
        "Total Operating Costs",
        "Total Personnel Costs"
    ])
].copy()

hcva_input = hcva_input.pivot_table(
    index=["year", "legal_entity"],
    columns="metric",
    values="value",
    aggfunc="sum"
).reset_index()

hcva_input.columns.name = None

hcva_input = hcva_input.rename(columns={
    "Net Banking Income (PNB)": "gnp",
    "Total Operating Costs": "operating_costs",
    "Total Personnel Costs": "Total Personnel Costs"
})

hcva_input.head()

,year,legal_entity,gnp,operating_costs,Total Personnel Costs
0,2023,O/W Europe,1.674701e+06,-1.181789e+06,-586929.04595
1,2023,Toutes Filiales Conso,1.677332e+06,-1.183365e+06,-601963.10454
2,2024,O/W Europe,2.083214e+06,-1.483243e+06,-731165.75369
3,2024,Toutes Filiales Conso,2.083437e+06,-1.480167e+06,-764928.35012
4,2025,O/W Europe,2.100417e+06,-1.443215e+06,-752170.61849


In [13]:
# HCVA computation
# ==========================================
# 1. CHARGEMENT ET NETTOYAGE DES FTE
# ==========================================

# Chargement du fichier
xls = pd.ExcelFile("AlbertSchool_CACEIS_PL-FTE_22-25_Sent.xlsx")
fte_raw = pd.read_excel(xls, sheet_name="Synthese_ETP")
fte_raw = clean_columns(fte_raw) # Ta fonction de nettoyage de colonnes

fte_tidy = fte_raw.copy()
col_entity = fte_tidy.columns[0]
col_2023 = fte_tidy.columns[5]
col_2024 = fte_tidy.columns[7]
col_2025 = fte_tidy.columns[9]

# Renommage et filtrage initial
fte_tidy = fte_tidy.rename(columns={
    col_entity: "legal_entity",
    col_2023: "fte_2023",
    col_2024: "fte_2024",
    col_2025: "fte_2025"
})

# Nettoyage des lignes (on garde ce qui n'est pas nul et on vire les headers)
fte_tidy = fte_tidy[fte_tidy["legal_entity"].notna()].copy()
fte_tidy = fte_tidy[
    ~fte_tidy["legal_entity"].astype(str).str.contains("employee|nan", case=False, na=False)
]

# Passage au format Long (Melt)
fte_df = fte_tidy.melt(
    id_vars="legal_entity",
    value_vars=["fte_2023", "fte_2024", "fte_2025"],
    var_name="year",
    value_name="fte"
)

# Conversion des types
fte_df["year"] = fte_df["year"].str.extract(r"(\d{4})").astype(int)
fte_df["fte"] = pd.to_numeric(fte_df["fte"], errors="coerce")

# Nettoyage spécifique des noms d'entités (Suppression des "TOTAL (", etc.)
fte_df["legal_entity"] = (
    fte_df["legal_entity"]
    .astype(str)
    .str.replace(r"TOTAL \(", "", regex=True)
    .str.replace(r"\)", "", regex=True)
    .str.strip()
)

print(fte_df.head())

# ==========================================
# 2. PRÉPARATION DE LA JOINTURE (NORMALISATION)
# ==========================================

def normalize_entity(x):
    x = str(x).strip().lower()
    x = x.replace("total", "")
    x = x.replace("(", "").replace(")", "")
    # On harmonise les termes spécifiques si besoin
    return x.strip()

# Création d'une clé de jointure commune 'entity_key'
# On applique cela sur le tableau financier (hcva_input) et le tableau RH (fte_df)
hcva_input["entity_key"] = hcva_input["legal_entity"].apply(normalize_entity)
fte_df["entity_key"] = fte_df["legal_entity"].apply(normalize_entity)

# ==========================================
# 3. FUSION DES DONNÉES (MERGE)
# ==========================================

# On supprime d'abord toute colonne fte vide existante par précaution
hcva_input = hcva_input.drop(columns=["fte"], errors="ignore")

# On fusionne pour récupérer les vrais FTE basés sur l'année et la clé normalisée
hcva_input = hcva_input.merge(
    fte_df[["year", "entity_key", "fte"]],
    on=["year", "entity_key"],
    how="left"
)

# ==========================================
# 4. CALCUL DU HCVA
# ==========================================

# S'assurer que les coûts sont en valeur absolue (positifs)
hcva_input["Total Personnel Costs"] = hcva_input["Total Personnel Costs"].abs()
hcva_input["operating_costs"] = hcva_input["operating_costs"].abs()

# Application de la formule : (PNB - (Coûts Opé - Coûts Personnel)) / FTE
# Cela revient à diviser la Valeur Ajoutée (PNB - Frais Généraux) par les effectifs
hcva_input["hcva"] = (
    hcva_input["gnp"] - 
    (hcva_input["operating_costs"] - hcva_input["Total Personnel Costs"])
) / hcva_input["fte"]

# Paramètre par défaut pour la direction si non spécifié
hcva_input["direction"] = "All Directions"

# Création du DataFrame final propre
hcva_df = hcva_input[["year", "legal_entity", "direction", "hcva"]]

# Affichage du résultat final
print(hcva_df)

            legal_entity  year      fte
0  Toutes Filiales conso  2023  6370.66
1             o/w Europe  2023  5338.66
2  Toutes Filiales conso  2024  6635.56
3             o/w Europe  2024  5459.56
4  Toutes Filiales conso  2025  6631.47
   year           legal_entity       direction        hcva
0  2023             O/W Europe  All Directions  202.268295
1  2023  Toutes Filiales Conso  All Directions  172.027706
2  2024             O/W Europe  All Directions  243.817601
3  2024  Toutes Filiales Conso  All Directions  206.191848
4  2025             O/W Europe  All Directions  259.770142
5  2025  Toutes Filiales Conso  All Directions  218.613420


In [14]:
# CONSTRUCTION DU DATAFRAME CENTRAL : kpi_dashboard_df
# =================================================================

# 1. Initialisation avec ta variable YEARS
kpi_dashboard_df = pd.DataFrame({'year': YEARS})

# 2. Intégration du HCVA
if 'hcva' in hcva_df.columns:
    # On s'assure de ne prendre que les données Conso pour la cohérence avec les autres KPIs
    hcva_filtered = hcva_df[hcva_df['legal_entity']=="Toutes Filiales Conso"]
    kpi_dashboard_df = kpi_dashboard_df.merge(
        hcva_filtered[['year', 'hcva']], 
        on='year', 
        how='left'
    )

kpi_dashboard_df

,year,hcva
0,2023,172.027706
1,2024,206.191848
2,2025,218.613420


### KTI computation

In [15]:
cold_review.head()


,date,matricule,formation,organization,session_id,date_de_début_de_session,date_de_fin_de_session,lieu_de_session,status,considérez_vous_que_cette_formation_vous_a_permis_de_prendre_confiance_en_vous_,...,considérez_vous_que_cette_formation_vous_a_permis_de_développer_de_nouvelles_compétences_,autres_à_préciser_,la_formation_visait_elle_la_préparation_d_un_diplôme_ou_d_une_certification_,si_oui_avez_vous_obtenu_le_diplôme_ou_la_certification_visé_e_,si_non_pourquoi_,la_formation_a_t_elle_répondu_à_vos_attentes_initiales_,estimez_vous_que_la_formation_était_en_adéquation_avec_le_métier_ou_les_réalités_du_secteur_,recommanderiez_vous_ce_stage_à_une_personne_exerçant_le_même_métier_que_vous_,utilisez_vous_les_connaissances_acquises_lors_de_la_formation_,quels_étaient_selon_vous_les_principaux_points_forts_et_les_principaux_axes_d_amélioration_de_cette_formation_
0,19/02/2026,ANON_76X90X48X48X57X49X50X55X55X48X53X,Formation FinOps : Maîtriser et optimiser ses ...,PLB CONSULTANT,25-FF-724-572,20/11/2025,21/11/2025,A distance,En attente,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19/02/2026,ANON_76X90X48X48X57X49X53X52X48X56X52X,"Bâle 3, quels enjeux pour les métiers de la fi...",IFCAM,25-B3Q-739-075,20/11/2025,21/11/2025,Paris,Complétée,Oui,...,Oui,NaN,Non,NaN,NaN,"Oui, tout à fait","Oui, tout à fait","Oui, tout à fait","Oui, en partie",NaN
2,18/02/2026,ANON_76X90X48X48X57X49X50X52X49X52X56X,"Parcours nouveaux managers - Suivre, piloter e...",ERYS,25-PNM-889-284,20/11/2025,20/11/2025,Montrouge,En attente,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,18/02/2026,ANON_76X90X48X48X57X49X49X55X49X56X57X,"Parcours nouveaux managers - Suivre, piloter e...",ERYS,25-PNM-889-284,20/11/2025,20/11/2025,Montrouge,Complétée,Oui,...,Oui,ok,Non,Non,ok,"Oui, en partie","Oui, en partie","Oui, en partie","Oui, en partie",NaN
4,18/02/2026,ANON_75X88X48X48X48X48X50X55X56X55X55X,"Parcours nouveaux managers - Suivre, piloter e...",ERYS,25-PNM-889-284,20/11/2025,20/11/2025,Montrouge,En attente,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
cold_review.columns.tolist()

['date',
 'matricule',
 'formation',
 'organization',
 'session_id',
 'date_de_début_de_session',
 'date_de_fin_de_session',
 'lieu_de_session',
 'status',
 'considérez_vous_que_cette_formation_vous_a_permis_de_prendre_confiance_en_vous_',
 'considérez_vous_que_cette_formation_vous_a_permis_de_faciliter_votre_quotidien_',
 'considérez_vous_que_cette_formation_vous_a_permis_d_améliorer_la_qualité_ou_l_efficacité_de_votre_travail_',
 'considérez_vous_que_cette_formation_vous_a_permis_de_vous_perfectionner_dans_un_domaine_que_vous_connaissiez_déjà_',
 'considérez_vous_que_cette_formation_vous_a_permis_de_développer_de_nouvelles_compétences_',
 'autres_à_préciser_',
 'la_formation_visait_elle_la_préparation_d_un_diplôme_ou_d_une_certification_',
 'si_oui_avez_vous_obtenu_le_diplôme_ou_la_certification_visé_e_',
 'si_non_pourquoi_',
 'la_formation_a_t_elle_répondu_à_vos_attentes_initiales_',
 'estimez_vous_que_la_formation_était_en_adéquation_avec_le_métier_ou_les_réalités_du_secteur_',
 'r

In [17]:
training.head()

,employee_code,entity,direction,attended_courses,organization,seesion_start_date,session_end_date,session_id,status,total_training_hours,certifications,year
0,NaN,NaN,NaN,Modèle de données GP4,NEOXAM,15/10/2025,17/10/2025,25-MDG-859-226,Réalisé,21.0,No,2025
1,NaN,CACEIS Bank,NaN,2024 International compliance trainings,IFCAM,29/11/2024,29/11/2024,NaN,Annulée,0.0,No,2024
2,NaN,CACEIS Bank,NaN,7Speaking plateforme licences annuelles,7Speaking,2025-06-01 00:00:00,2025-06-01 00:00:00,24-7PL-451-669,Réalisé,0.5,No,2025
3,NaN,CACEIS Bank,NaN,7Speaking plateforme licences annuelles,7Speaking,15/12/2023,15/12/2023,23-7PL-451-343,Réalisé,0.1,NaN,2023
4,NaN,CACEIS Bank,NaN,Accompagnement à la négociation commerciale,Dale Carnegie,NaN,NaN,NaN,Annulée,14.0,No,2025


In [18]:

# =================================================================
# 1. FONCTION DE SCORING (TRANSFORMATION DU TEXTE EN VALEUR ROI)
# =================================================================
def score_utilization(val):
    """
    Transforme les réponses qualitatives en scores numériques (ROI).
    1.0 = Succès | 0.5 = Partiel | 0.2 = Marginal (Pas vraiment) | 0.0 = Nul
    """
    val = str(val).lower().strip()
    
    if val in ['nan', '', 'none']:
        return np.nan
    
    # 1. Le cas spécifique "Pas vraiment" (Prioritaire)
    if 'pas vraiment' in val:
        return 0.2
    
    # 2. Succès : Utilisation claire et régulière
    if any(keyword in val for keyword in ['tout à fait', 'régulièrement', 'oui', 'souvent']):
        return 1.0
        
    # 3. Mitigé : Utilisation partielle
    if any(keyword in val for keyword in ['partiellement', 'partie', 'temps en temps', 'un peu']):
        return 0.5
        
    # 4. Échec : "Non", "Pas du tout", ou autre réponse négative
    return 0.0

# =================================================================
# 2. PRÉPARATION DES DONNÉES (CLEANING)
# =================================================================

# Nettoyage Cold Review
cold_clean = cold_review.copy()
cold_clean['matricule'] = cold_clean['matricule'].astype(str).str.strip()
# On s'assure d'avoir l'année (si non présente, on l'extrait de la date)
if 'year' not in cold_clean.columns:
    cold_clean['year'] = pd.to_datetime(cold_clean['date'], dayfirst=True, errors='coerce').dt.year

# Application du score de transfert de compétences
target_col = "utilisez_vous_les_connaissances_acquises_lors_de_la_formation_"
cold_clean['utilization_score'] = cold_clean[target_col].apply(score_utilization)

# Nettoyage Training
training_clean = training.copy()
training_clean['employee_code'] = training_clean['employee_code'].astype(str).str.strip()
if 'year' not in training_clean.columns:
    training_clean['year'] = pd.to_datetime(training_clean['date_de_fin_de_session'], dayfirst=True, errors='coerce').dt.year

# =================================================================
# 3. JOINTURE ET CALCUL DU ROI (KTI)
# =================================================================

# On part du fichier Training (Investissement) vers la Cold Review (Résultat)
training_clean.columns = [c.strip() for c in training_clean.columns]
cold_clean.columns = [c.strip() for c in cold_clean.columns]
# Jointure par Matricule et Session pour une précision maximale
kti_merged = training_clean.merge(
    cold_clean[['matricule', 'session_id', 'utilization_score']], 
    left_on=['employee_code', 'session_id'], 
    right_on=['matricule', 'session_id'], 
    how='left'
)

# Agrégation par Entité et Direction
kti_final = kti_merged.groupby(['year', 'entity', 'direction']).agg(
    total_formations=('employee_code', 'count'),      # Budget investi (nb de sessions)
    nb_reponses_encaissees=('utilization_score', 'count'), # Preuves de retour (Cold Reviews)
    somme_activation_savoir=('utilization_score', 'sum')   # Somme des scores 1, 0.5, 0
).reset_index()

# --- CALCUL DES COMPOSANTES DU KTI ---

# 1. Utilization Score (Efficacité pédagogique)
# % de savoir activé parmi ceux qui ont répondu
kti_final['utilization_rate'] = (
    kti_final['somme_activation_savoir'] / kti_final['nb_reponses_encaissees']
).fillna(0)

# 2. Response Rate (Solidité de la preuve)
# % de formés ayant fait leur évaluation à froid
kti_final['response_rate'] = (
    kti_final['nb_reponses_encaissees'] / kti_final['total_formations']
).fillna(0)

# 3. KTI FINAL (Indicateur de ROI global)
kti_final['kti'] = kti_final['utilization_rate'] * kti_final['response_rate']

# =================================================================
# 4. FINALISATION POUR LE DASHBOARD PDG
# =================================================================
kti_df = kti_final.rename(columns={'entity': 'legal_entity'})

# On calcule le KTI si le taux de réponse était de 100%
kti_df['kti_potentiel'] = kti_df['utilization_rate'] * 1.0  # (car response_rate serait 1.0)

# On calcule l'écart (le ROI "caché" ou "perdu" par manque de données)
kti_df['gap_roi'] = kti_df['kti_potentiel'] - kti_df['kti']

# On ne garde que les colonnes essentielles pour la clarté
kti_dashboard = kti_df[[
    'year', 'legal_entity', 'direction', 
    'utilization_rate', 'response_rate', 'kti', 'kti_potentiel', 'gap_roi'
]]

# Tri pour voir les meilleures entités en haut
kti_dashboard = kti_dashboard.sort_values(by='kti', ascending=False)

print("KTI (ROI Formation) prêt pour présentation :")
display(kti_dashboard.sample(10))

KTI (ROI Formation) prêt pour présentation :


,year,legal_entity,direction,utilization_rate,response_rate,kti,kti_potentiel,gap_roi
47,2023,CACEIS Bank,SPF - Client Port Compliance,0.782759,0.297436,0.232821,0.782759,0.549938
146,2025,CACEIS Bank,COV - General Secretary,0.740000,0.476190,0.352381,0.740000,0.387619
122,2024,CACEIS Fund Administration,SPF - Legal,1.000000,0.315789,0.315789,1.000000,0.684211
1,2023,CACEIS,BUT - Inf System Sec & Resil,1.000000,1.000000,1.000000,1.000000,0.000000
119,2024,CACEIS Fund Administration,COV - PERES,0.808511,0.225962,0.182692,0.808511,0.625818
93,2024,CACEIS Bank,COV - Client & Bus Dev Support,0.778125,0.249027,0.193774,0.778125,0.584351
23,2023,CACEIS Bank,BUT - Fund Services,0.793258,0.270517,0.214590,0.793258,0.578669
131,2025,CACEIS,SPF - Fin Treasury & Admin,0.866667,0.117647,0.101961,0.866667,0.764706
69,2024,CACEIS,BUT - Market Solutions,0.000000,0.000000,0.000000,0.000000,0.000000
54,2023,CACEIS Bank,STI - CACEIS Consulting,0.813953,0.279221,0.227273,0.813953,0.586681


In [19]:
# =================================================================
# 5. AGRÉGATION ANNUELLE POUR LE GRAPHIQUE STREAMLIT (VERSION ROI)
# =================================================================

# Agrégation des sommes par année
kti_yearly = kti_df.groupby('year').agg(
    total_formations=('total_formations', 'sum'),
    nb_reponses=('nb_reponses_encaissees', 'sum'),
    somme_activation=('somme_activation_savoir', 'sum')
).reset_index()

# 1. Calcul du KTI Réel (basé sur les faits prouvés)
kti_yearly['utilization_rate'] = (kti_yearly['somme_activation'] / kti_yearly['nb_reponses']).fillna(0)
kti_yearly['response_rate'] = (kti_yearly['nb_reponses'] / kti_yearly['total_formations']).fillna(0)
kti_yearly['kti'] = kti_yearly['utilization_rate'] * kti_yearly['response_rate']

# 2. Calcul du KTI Potentiel (Si on avait 100% de réponses)
# Le potentiel est simplement égal au taux d'utilisation réel constaté sur le terrain
kti_yearly['kti_potentiel'] = kti_yearly['utilization_rate']

# 3. Calcul du Gap ROI (La valeur "cachée" non capturée par l'enquête)
kti_yearly['gap_roi'] = kti_yearly['kti_potentiel'] - kti_yearly['kti']

# 4. Préparation des données pour le graphique Streamlit
kti_trend = kti_yearly[['year', 'kti', 'kti_potentiel']]

# Affichage pour vérification
print("Tableau de bord de tendance ROI :")
display(kti_trend)

Tableau de bord de tendance ROI :


,year,kti,kti_potentiel
0,2023,0.185466,0.827397
1,2024,0.230899,0.840612
2,2025,0.157579,0.857371


In [20]:
#  Intégration du KTI et ses métriques
if not kti_trend.empty:
    kpi_dashboard_df = kpi_dashboard_df.merge(
        kti_trend, 
        on='year', 
        how='left'
    )

kpi_dashboard_df

,year,hcva,kti,kti_potentiel
0,2023,172.027706,0.185466,0.827397
1,2024,206.191848,0.230899,0.840612
2,2025,218.613420,0.157579,0.857371


### Resilience & Engagement (RE score)

In [21]:
# ============================================================
# 1. PARAMÉTRAGE ET FILTRES
# ============================================================

# Liste des fichiers d'absentéisme 
# Chargement avec les noms d'onglets que tu as précisés
abs_2023_raw = absenteeism_2023.copy()
abs_2024_raw = absenteeism_2024.copy()
abs_2025_raw = absenteeism_2025.copy()

print("Absenteeism 2025 columns:", abs_2025_raw.columns.tolist())
print("Absenteeism 2024 columns:", abs_2024_raw.columns.tolist())
print("Absenteeism 2023 columns:", abs_2023_raw.columns.tolist())

# Liste des motifs d'absence à inclure selon votre demande
MOTIFS_CIBLES = [
    "Absences Autres motifs", 
    "Absences non suivi", 
    "Maladie", 
    "Accident", 
    "Absences non autorisée"
]

def process_absenteeism(df, year):
    # Mapping des noms de colonnes pour la jointure
    # On transforme 'société' en 'legal_entity' et 'niveau 6' en 'direction'
    mapping = {
        'société': 'legal_entity',
        'niveau_6': 'direction'
    }
    # On passe tout en minuscule pour être sûr que le mapping matche
    df.columns = df.columns.str.lower()
    df = df.rename(columns=mapping)
    
    # Identification des colonnes 
    col_regroupement = "regroupement_jour_absences"
    col_jours = "jour_calendaires_absence"
    
    # Filtrage sur les motifs cibles
    df_filtered = df[df[col_regroupement].isin(MOTIFS_CIBLES)].copy()
    
    # Conversion numérique (gestion des demi-journées 0,5)
    if df_filtered[col_jours].dtype == 'object':
        df_filtered[col_jours] = (
            df_filtered[col_jours]
            .astype(str)
            .str.replace(',', '.')
            .astype(float)
        )
    
    # Ajout de l'année pour la consolidation
    df_filtered['year'] = year
    return df_filtered

# Application du traitement
abs_2023_clean = process_absenteeism(abs_2023_raw, 2023)
abs_2024_clean = process_absenteeism(abs_2024_raw, 2024)
abs_2025_clean = process_absenteeism(abs_2025_raw, 2025)

# ============================================================
# 3. CONSOLIDATION ET SYNTHÈSE
# ============================================================

abs_all_years = pd.concat([abs_2023_clean, abs_2024_clean, abs_2025_clean], ignore_index=True)

# Agrégation par année, entité et direction pour le dashboard
abs_summary = abs_all_years.groupby(['year', 'legal_entity', 'direction']).agg(
    total_jours_absence=("jour_calendaires_absence", 'sum')
).reset_index()

print("Fichiers chargés par onglets et filtrés avec succès.")
display(abs_summary.head())

Absenteeism 2025 columns: ['mat_compliance', 'employee_code', 'nom', 'prénom', 'genre', 'type_contrat', 'contrat_particulier', '_concat_contrat', 'présents_non_présents', 'code_régime_temps_travail', 'régime_temps_travail', 'régime_temps_travail_', 'code_société', 'société', 'code_niveau_6', 'niveau_6', 'code_niveau_7', 'niveau_7', 'code_niveau_8', 'niveau_8', 'code_niveau_9', 'niveau_9', 'code_organisation_1', 'organisation_1', 'code_organisation_2', 'organisation_2', 'code_organisation_3', 'organisation_3', 'code_organisation_4', 'organisation_4', 'code_organisation_5', 'organisation_5', 'code_organisation_6', 'organisation_6', 'code_organisation_7', 'organisation_7', 'code_organisation_8', 'organisation_8', 'date_absence', 'aaaa_mm_absence', 'code_motif_jour_absence', 'motif_jour_absence', 'regroupement_jour_absences', 'jour_calendaires_absence', 'jours_ouvrables_absence', 'jours_ouvrés_absence']
Absenteeism 2024 columns: ['unnamed_0', 'employee_code', 'nom', 'prénom', 'genre', 'typ

,year,legal_entity,direction,total_jours_absence
0,2023,CACEIS,CLIENT & BUSINESS DEVELOPMENT SUPPORT,129.0
1,2023,CACEIS,COMPLIANCE,157.0
2,2023,CACEIS,FINANCE AND ADMINISTRATION,1854.0
3,2023,CACEIS,GENERAL INSPECTION,630.0
4,2023,CACEIS,GENERAL MANAGEMENT,25.5


In [22]:
# ============================================================
# CALCUL DU TAUX D'ABSENTÉISME GLOBAL
# ============================================================

# 1. Constante
NB_JOURS_OUVRES = 252

# 2. Préparation des FTE : on filtre une seule fois pour toutes les années
fte_conso = fte_df[fte_df['legal_entity'] == 'Toutes Filiales conso'][['year', 'fte']]

# 3. Préparation des Absences : on agrège par année (somme globale toutes entités confondues)
abs_yearly_total = abs_summary.groupby('year')['total_jours_absence'].sum().reset_index()

# 4. Fusion des deux métriques
abs_rate_calc = abs_yearly_total.merge(fte_conso, on='year', how='inner')

# 5. Calcul vectorisé (plus rapide que ligne par ligne)
abs_rate_calc['absenteeism_rate'] = (
    abs_rate_calc['total_jours_absence'] / (NB_JOURS_OUVRES * abs_rate_calc['fte'])
) * 100

# 6. Affichage dynamique
for _, row in abs_rate_calc.iterrows():
    print(f"Taux d'absentéisme pour {int(row['year'])} : {row['absenteeism_rate']:.2f}%")


Taux d'absentéisme pour 2023 : 8.74%
Taux d'absentéisme pour 2024 : 1.53%
Taux d'absentéisme pour 2025 : 1.61%


In [23]:
# ============================================================
# 1. CRÉATION DU TABLEAU D'ENGAGEMENT (DONNÉES EXTRAITES)
# ============================================================

# On crée un dictionnaire avec les scores extraits des fichiers IMR et TI
# Note : Assure-toi que ces scores sont ramenés sur une échelle de 5
data_engagement = {
    'year': [2023, 2024, 2025],
    'source_file': ['IMR 2023', 'IMR 2024', 'TI 2025'],
    'engagement_score': [3.8, 3.85, 3.85]  # TI indice de confiance 2025=0.77/20; IMR Accountability Index IMR 2024=0.77/20; IMR 2023=0.76/20 IMR Accountability Index
}


engagement_df = pd.DataFrame(data_engagement)

In [24]:
# ============================================================
# 1. CALCUL DU RE-SCORE FINAL
# ============================================================

# On part du tableau abs_rate_calc (qui contient déjà absenteeism_rate par année)
# On y ajoute les scores d'engagement pour faire le calcul
re_score_final = abs_rate_calc[['year', 'absenteeism_rate']].merge(
    engagement_df[['year', 'engagement_score']], 
    on='year', 
    how='left'
)

# Calcul du RE-Score (Formule : Engagement * Taux de Présence)
# Note : absenteeism_rate est divisé par 100 s'il est en pourcentage (ex: 4.5)
re_score_final['re_score'] = (
    re_score_final['engagement_score'] * (1 - (re_score_final['absenteeism_rate'] / 100))
)

# ============================================================
# 2. ALIMENTATION DU DASHBOARD FINAL
# ============================================================

# On injecte uniquement le RE-Score dans le dashboard principal
kpi_dashboard_df['re_score'] = kpi_dashboard_df['year'].map(
    re_score_final.set_index('year')['re_score']
)
print("Dashboard final avec RE-Score intégré :")
display(kpi_dashboard_df)   

Dashboard final avec RE-Score intégré :


,year,hcva,kti,kti_potentiel,re_score
0,2023,172.027706,0.185466,0.827397,3.467987
1,2024,206.191848,0.230899,0.840612,3.790916
2,2025,218.613420,0.157579,0.857371,3.788035


### CHHI Computation

In [30]:

# Targets
TARGETS = {
    "HCVA": 200,
    "KTI": 0.75,
    "Skill_Decay": 0.12,
    "RE_Score": 4,
    "SPE": 0.25
}

# Weights
WEIGHTS = {
    "HCVA": 0.40,
    "KTI": 0.30,
    "RE_Score": 0.30
}

# Weights
#WEIGHTS = {
#    "HCVA": 0.30,
#    "KTI": 0.20,
#    "Skill_Decay": 0.20,
#    "RE_Score": 0.15,
#    "SPE": 0.15
#}

In [ ]:
# ============================================================
# NORMALISATION FINALE 
# ============================================================

# 1. Normalisation HCVA 
kpi_dashboard_df['hcva_normalized'] = (kpi_dashboard_df['hcva'] / TARGETS["HCVA"]) * 5

# 2. Normalisation KTI Potentiel
# On suppose que kti_potentiel est un ratio (ex: 0.8 pour 80%)
kpi_dashboard_df['kti_normalized'] = (kpi_dashboard_df['kti_potentiel']/ TARGETS["KTI"]) * 5

# 3. RE-Score (Déjà sur 5)
kpi_dashboard_df['re_normalized'] = (kpi_dashboard_df['re_score'] / TARGETS["RE_Score"]) * 5

# ============================================================
# CALCUL DU CHHI PONDÉRÉ
# ============================================================

kpi_dashboard_df['chhi_score'] = (
    WEIGHTS["HCVA"] * kpi_dashboard_df['hcva_normalized'] +
    WEIGHTS["KTI"] * kpi_dashboard_df['kti_normalized'] +
    WEIGHTS["RE_Score"] * kpi_dashboard_df['re_normalized'] 
)

#kpi_dashboard_df['chhi_score'] = (
#    0.30 * kpi_dashboard_df['hcva_normalized'] +
#    0.20 * kpi_dashboard_df['kti_normalized'] +
#    0.20 * kpi_dashboard_df['skill_score'] +
#    0.15 * kpi_dashboard_df['re_normalized'] +
#    0.15 * kpi_dashboard_df['spe_score']
#)

# Arrondi pour la lisibilité
kpi_dashboard_df['chhi_score'] = kpi_dashboard_df['chhi_score'].round(2)

print(f"CHHI calculé avec les poids suivants : HCVA={WEIGHTS['HCVA']*100}%, KTI={WEIGHTS['KTI']*100}%, RE-Score={WEIGHTS['RE_Score']*100}%")
display(kpi_dashboard_df[['year', 'hcva', 'hcva_normalized', 'chhi_score']])

CHHI calculé avec les poids suivants : HCVA=40.0%, KTI=30.0%, RE-Score=30.0%


,year,hcva,hcva_normalized,chhi_score
0,2023,172.03,4.30075,4.68
1,2024,206.19,5.15475,5.16
2,2025,218.61,5.46525,5.33


In [32]:
# ============================================================
# CALCUL DU CHHI SUR BASE 100 (POUR RADAR CHART)
# ============================================================

# Note : On multiplie simplement les scores normalisés par 20 (car 5 * 20 = 100)
# Cela garantit que 100% sur le radar = Cible atteinte. 

# HCVA
kpi_dashboard_df['hcva_radar'] = kpi_dashboard_df['hcva_normalized'] * 20

# KTI 
kpi_dashboard_df['kti_radar'] = kpi_dashboard_df['kti_normalized'] * 20

# RE-Score
kpi_dashboard_df['re_radar'] = kpi_dashboard_df['re_normalized']  * 20

# On peut aussi calculer le score global sur 100 pour l'affichage
kpi_dashboard_df['chhi_index_100'] = kpi_dashboard_df['chhi_score'] * 20

In [33]:
# Nettoyage final
kpi_dashboard_df = kpi_dashboard_df.fillna(0)

# Arrondir toutes les colonnes numériques à 2 décimales
kpi_dashboard_df = kpi_dashboard_df.round(2)

# =================================================================
# EXPORTATION
# =================================================================
kpi_dashboard_df.to_csv("dashboard_kpi_hr.csv", index=False, sep=';', encoding='utf-8-sig')

print(f"Le dashboard pour les années {YEARS} est prêt !")
display(kpi_dashboard_df)

Le dashboard pour les années [2023, 2024, 2025] est prêt !


,year,hcva,kti,kti_potentiel,re_score,hcva_normalized,kti_normalized,re_normalized,chhi_score,hcva_radar,kti_radar,re_radar,chhi_index_100
0,2023,172.03,0.19,0.83,3.47,4.30,5.53,4.34,4.68,86.02,110.67,86.75,93.6
1,2024,206.19,0.23,0.84,3.79,5.15,5.60,4.74,5.16,103.10,112.00,94.75,103.2
2,2025,218.61,0.16,0.86,3.79,5.47,5.73,4.74,5.33,109.30,114.67,94.75,106.6
